# SiC Wafer Dicing Simulation — End-to-End Demo

**Pipeline**: ABAQUS FEM → GP Surrogate → Bayesian Optimization → TMCMC Inference

This notebook demonstrates the full workflow for optimizing SiC blade dicing parameters
using physics-based simulation and machine learning.

| Stage | Tool | Output |
|-------|------|--------|
| 1. FEM | ABAQUS/Explicit | Chipping fraction, stress field |
| 2. Surrogate | Gaussian Process | Response surface |
| 3. BO | Expected Improvement | Optimal parameters |
| 4. Inference | TMCMC | Posterior distribution |

**Material**: 4H-SiC  
**Parameters**: Cut depth [10–70 µm], Blade width [15–50 µm]

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib import cm

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

## 1. Material Properties

In [ ]:
from data.materials.material_properties import SiC, Si, GaN

print("=== 4H-SiC ===")
for k, v in SiC.items():
    print(f"  {k:20s}: {v}")

print("\n=== Fracture energy G_c ===")
for mat in [Si, SiC, GaN]:
    Gc = mat['K_Ic']**2 / mat['E']
    print(f"  {mat['name']:12s}: G_c = {Gc:.4f} J/m²")

## 2. Experimental Data (Micro2026 + Mat2022)

Digitized from open-access publications:
- **[Micro2026]** Micromachines 17(2):187, 2026 — DOI:10.3390/mi17020187
- **[Mat2022]** Materials 15(22):8083, 2022 — DOI:10.3390/ma15228083

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

from validation.experimental_data import CHIPPING_DATA, QUALITATIVE_TRENDS

df = pd.DataFrame(CHIPPING_DATA)
print(f"Loaded {len(df)} data points from {df['source'].nunique()} sources")
print(df[['source','cut_depth_um','blade_W_um','feed_mm_s','spindle_rpm','chipping_um']].to_string(index=False))

In [ ]:
# Visualize experimental data: three 1D sweeps
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
colors = {'Micro2026': '#2166ac', 'Mat2022': '#d6604d'}

sweep_axes = [
    (axes[0], 'cut_depth_um',  'Cut Depth [µm]'),
    (axes[1], 'feed_mm_s',     'Feed Speed [mm/s]'),
    (axes[2], 'spindle_rpm',   'Spindle Speed [rpm]'),
]
for ax, (x_col, xlabel) in [(a, (c, l)) for a, (c, l) in zip([axes[0],axes[1],axes[2]],
    [('cut_depth_um','Cut Depth [µm]'),('feed_mm_s','Feed Speed [mm/s]'),('spindle_rpm','Spindle Speed [rpm]')])]:
    for src, grp in df.groupby('source'):
        ax.scatter(grp[x_col], grp['chipping_um'],
                   color=colors[src], s=70, edgecolors='k', lw=0.6, label=src)
    ax.axhline(15, color='red', ls='--', lw=1.2, label='15 µm threshold')
    ax.set_xlabel(xlabel); ax.set_ylabel('Front Chipping [µm]'); ax.legend(fontsize=9); ax.grid(alpha=0.3)

plt.suptitle('Experimental Chipping Data: 4H-SiC Blade Dicing', fontsize=12)
plt.tight_layout(); plt.show()

## 3. GP Surrogate — Trained on Experimental Data

4-feature GP:  → 

In [ ]:
from ml.train_from_experimental import ExperimentalGPSurrogate, FEATURE_COLS, TARGET_COL, REF

X = df[FEATURE_COLS].values.astype(float)
y = df[TARGET_COL].values.astype(float)

model = ExperimentalGPSurrogate()
print("[*] LOO cross-validation …")
cv = model.loo_cv(X, y)

print("
[*] Fitting on full dataset …")
model.fit(X, y)
print("GP fitted successfully")

In [ ]:
# 1D sweep predictions with uncertainty bands
from ml.train_from_experimental import _sweep_array

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
colors = {'Micro2026': '#2166ac', 'Mat2022': '#d6604d'}

sweeps = [
    ('cut_depth_um',  np.linspace(60, 420, 120),    'Cut Depth [µm]'),
    ('feed_mm_s',     np.linspace(0.3, 3.2, 100),   'Feed Speed [mm/s]'),
    ('spindle_rpm',   np.linspace(18000, 42000, 100),'Spindle Speed [rpm]'),
]

for ax, (vary_feat, x_vals, xlabel) in zip(axes, sweeps):
    fixed = {f: REF[f] for f in FEATURE_COLS if f != vary_feat}
    X_sw  = _sweep_array(vary_feat, x_vals, fixed)
    mu, sigma = model.predict(X_sw, return_std=True)

    ax.plot(x_vals, mu, color='#1a1a2e', lw=2, label='GP mean')
    ax.fill_between(x_vals, mu-2*sigma, mu+2*sigma, alpha=0.18, color='#1a1a2e', label='±2σ')
    for src, grp in df.groupby('source'):
        ax.scatter(grp[vary_feat], grp[TARGET_COL],
                   color=colors[src], s=60, edgecolors='k', lw=0.6, zorder=5, label=src)
    ax.axhline(15, color='#d73027', ls='--', lw=1.2, label='15 µm threshold')
    ax.set_xlabel(xlabel); ax.set_ylabel('Front Chipping [µm]')
    ax.legend(fontsize=8); ax.grid(alpha=0.25)

plt.suptitle('GP Surrogate — 1D Sweep (others fixed at Micro2026 reference)', fontsize=12)
plt.tight_layout(); plt.show()

## 4. Response Surface — Depth × Feed

Fixed: blade_W = 23 µm, spindle = 30,000 rpm

In [ ]:
depths = np.linspace(60, 420, 80)
feeds  = np.linspace(0.3, 3.2, 80)
D, F   = np.meshgrid(depths, feeds)
X_grid = np.column_stack([D.ravel(), np.full(D.size, 23.0), F.ravel(), np.full(D.size, 30000.0)])

mu, sigma = model.predict(X_grid, return_std=True)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, data, title, cmap in [
    (axes[0], mu,    'GP Mean — Chipping [µm]', 'YlOrRd'),
    (axes[1], sigma, 'GP Std [µm]',              'viridis'),
]:
    im = ax.contourf(D, F, data.reshape(D.shape), levels=20, cmap=cmap)
    plt.colorbar(im, ax=ax)
    ax.contour(D, F, mu.reshape(D.shape), levels=[15.0], colors='white', lw=1.5, linestyles='--')
    ax.scatter(df['cut_depth_um'], df['feed_mm_s'], c='white', s=50, edgecolors='k', zorder=5)
    ax.set_xlabel('Cut Depth [µm]'); ax.set_ylabel('Feed Speed [mm/s]'); ax.set_title(title)

plt.suptitle('4H-SiC Blade Dicing — Chipping Response Surface
(blade_W=23µm, spindle=30krpm; dashed = 15µm threshold)', fontsize=11)
plt.tight_layout(); plt.show()
print(f"Safe region (chipping<15µm): depth<{depths[mu.reshape(D.shape).mean(axis=0)<15].max():.0f}µm at mean feed")

## 5. TMCMC Inference

Given observed chipping, infer (cut_depth, feed_speed) distribution via Bayesian inference.

In [ ]:
from optimization.tmcmc_dicing import calibrate_experimental

# Scenario 1: observed chipping 10µm (Micro2026 reference: depth=390µm, feed=1mm/s)
result = calibrate_experimental(
    observed_chip_um=10.0,
    blade_W_um=23.0,
    spindle_rpm=30000.0,
    n_samples=600,
)

import json
print(json.dumps(result, indent=2))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

samples = np.load('results/tmcmc_exp_calibrate_samples.npy')
weights = np.load('results/tmcmc_exp_calibrate_weights.npy')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
sc = ax.scatter(samples[:, 0], samples[:, 1], c=weights, cmap='plasma', s=15, alpha=0.7)
plt.colorbar(sc, ax=ax, label='Posterior weight')
ax.scatter(result['mean_cut_depth_um'], result['mean_feed_mm_s'],
           c='lime', s=200, marker='*', zorder=5, label='Posterior mean')
ax.set_xlabel('Cut Depth [µm]'); ax.set_ylabel('Feed Speed [mm/s]')
ax.set_title(f'TMCMC Posterior
observed chipping = 10.0 µm')
ax.legend(fontsize=9)

ax = axes[1]
ax.hist(samples[:, 0], bins=25, weights=weights, density=True,
        alpha=0.6, label='Cut depth [µm]', color='#2166ac')
ax2 = ax.twinx()
ax2.hist(samples[:, 1], bins=25, weights=weights, density=True,
         alpha=0.5, color='#d6604d', label='Feed [mm/s]')
ax.set_xlabel('Parameter value'); ax.set_title('Posterior Marginals')
ax.legend(loc='upper left', fontsize=9); ax2.legend(loc='upper right', fontsize=9)

print(f"MAP: depth={result['map_cut_depth_um']:.1f}µm, feed={result['map_feed_mm_s']:.3f}mm/s")
print(f"True values (Micro2026 ref): depth=390µm, feed=1.0mm/s")
plt.tight_layout(); plt.show()

## 6. Summary

| Component | Method | Key result |
|-----------|--------|------------|
| Data | Digitized from Micro2026 + Mat2022 | 18 data points, 4 features |
| Surrogate | 4-feature anisotropic RBF-GP | LOO-RMSE reported above |
| Response surface | GP prediction + uncertainty | Safe region: depth < threshold |
| Inference | TMCMC (Ching & Chen 2007) | Full posterior over (depth, feed) |

**Dominant effects** (consistent with literature):
- depth > feed > spindle (chipping sensitivity ranking)
- Critical threshold: chipping < 15 µm for production

**Next steps**:
1. Replace experimental data with real ABAQUS output once STATUS/RF extraction is fixed
2. Add GaN material data and extend feature space
3. Active learning: GP-guided FEM experiment selection